In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.ensemble import IsolationForest
from sklearn.exceptions import InconsistentVersionWarning
from sklearn.neighbors import LocalOutlierFactor
from torch import nn

In [2]:
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
N_TEST = 2405
N_TICKERS = 13
N_TEST_DATES = 185
TEST_FIRST_DATE = '2025-12-24'
TEST_LAST_DATE = '2026-09-25'
MODEL_ORDER = ('IF', 'AE', 'LOF')
TOLERANCES = {'IF': 1e-5, 'AE': 5e-4, 'LOF': 1e-5}

def sha256_file(path):
    hash_obj = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

# Notebook VS Code đôi khi chạy từ cwd khác với vị trí file. Chỉ chấp nhận
# thư mục có đầy đủ dữ liệu và model của Bước 03–07.
start_candidates = [Path.cwd()]
if '__file__' in globals():
    start_candidates.insert(0, Path(__file__).resolve().parent)
for p in list(start_candidates):
    start_candidates.extend([p / 'notebooks', p.parent, p.parent / 'notebooks'])
ROOT = None
for root in dict.fromkeys(p.resolve() for p in start_candidates):
    if (root / 'outputs' / '07_threshold_calibration_A' / '07_thresholds.json').is_file() or (
        root / '07_thresholds(1).json').is_file():
        if (root / 'outputs' / '03_feature_engineering' / '03_features_all.csv').is_file() or (
            root / '03_features_all.csv').is_file():
            ROOT = root
            break
if ROOT is None:
    raise FileNotFoundError(
        'Không tìm thấy gốc Stocks/notebooks có outputs Bước 03 và Bước 07. '
        f'CWD hiện tại: {Path.cwd()}.'
    )

# Nếu có nhiều bản copy cùng tên, buộc chọn tệp có SHA-256 khớp manifest.
def find_file(label, candidates, expected_sha=None):
    candidates = [ROOT / x for x in candidates]
    existent = [p for p in candidates if p.is_file()]
    if not existent:
        raise FileNotFoundError(f'Thiếu {label}. Đã tìm tại: ' + ', '.join(map(str, candidates)))
    if expected_sha:
        matched = [p for p in existent if sha256_file(p) == expected_sha]
        if not matched:
            raise ValueError(f'{label}: không tệp nào khớp SHA-256 đã khóa. Có thể nhầm phiên bản.')
        return matched[0]
    return existent[0]

POLICY_PATH = find_file('frozen policy P0', [
    '07_A_P0_frozen_policy(1).json',
    'outputs/07_P0_final_pretest_A/07_A_P0_frozen_policy.json',
    '07_A_P0_frozen_policy.json'
])
policy = json.loads(POLICY_PATH.read_text(encoding='utf-8-sig'))
assert policy['experiment'] == 'EXP1_A_P0'
assert policy['status'] == 'READY_FOR_SEPARATE_TEST_EVALUATION'
assert policy['model_features'] == FEATURES
assert policy['feature_window'] == 60
assert policy['calibration_quantile'] == 0.99
assert policy['test_scored'] is False
assert policy['models_refit'] is False
assert policy['proxy_name'] == 'P0_final_std20_mean20_OR'

GATE_PATH = find_file('pre-test gate', [
    '07_pretest_gate(1).json',
    'outputs/07_P0_final_pretest_A/07_pretest_gate.json',
    '07_pretest_gate.json'
])
gate = json.loads(GATE_PATH.read_text(encoding='utf-8-sig'))
required_gate_flags = [
    'data_03_ready_and_chronological', 'proxy_valid_all_model_ready_rows',
    'proxy_past_only', 'checkpoint_files_match_step07_hash',
    'checkpoint_scores_reproduce_step065', 'step07_q99_reproduced_without_refit',
    'q99_unchanged', 'test_open_authorized'
]
for flag in required_gate_flags:
    assert gate.get(flag) is True, f'Pre-test gate chưa đạt: {flag}'
assert gate['test_scores_computed'] is False

THRESH_PATH = find_file('07_thresholds.json', [
    'outputs/07_threshold_calibration_A/07_thresholds.json',
    '07_thresholds(1).json', '07_thresholds.json'
], policy['step07_threshold_manifest_sha256'])
threshold_manifest = json.loads(THRESH_PATH.read_text(encoding='utf-8-sig'))
assert threshold_manifest['quantile'] == 0.99
assert threshold_manifest['rule'] == 'score > threshold'
assert threshold_manifest['n_calibration'] == 962
assert threshold_manifest['feature_cols'] == FEATURES
THRESHOLDS = {m: float(threshold_manifest['thresholds'][m]['threshold']) for m in MODEL_ORDER}
for m in MODEL_ORDER:
    assert np.isclose(THRESHOLDS[m], policy['thresholds'][m], rtol=0, atol=1e-12)

In [3]:
FEATURES = ['z_gap', 'z_intraday', 'z_range', 'z_volume']
N_TEST = 2405
N_TICKERS = 13
N_TEST_DATES = 185
TEST_FIRST_DATE = '2025-12-24'
TEST_LAST_DATE = '2026-09-25'
MODEL_ORDER = ('IF', 'AE', 'LOF')
TOLERANCES = {'IF': 1e-5, 'AE': 5e-4, 'LOF': 1e-5}

def sha256_file(path):
    hash_obj = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            hash_obj.update(chunk)
    return hash_obj.hexdigest()

# Notebook VS Code đôi khi chạy từ cwd khác với vị trí file. Chỉ chấp nhận
# thư mục có đầy đủ dữ liệu và model của Bước 03–07.
start_candidates = [Path.cwd()]
if '__file__' in globals():
    start_candidates.insert(0, Path(__file__).resolve().parent)
for p in list(start_candidates):
    start_candidates.extend([p / 'notebooks', p.parent, p.parent / 'notebooks'])
ROOT = None
for root in dict.fromkeys(p.resolve() for p in start_candidates):
    if (root / 'outputs' / '07_threshold_calibration_A' / '07_thresholds.json').is_file() or (
        root / '07_thresholds(1).json').is_file():
        if (root / 'outputs' / '03_feature_engineering' / '03_features_all.csv').is_file() or (
            root / '03_features_all.csv').is_file():
            ROOT = root
            break
if ROOT is None:
    raise FileNotFoundError(
        'Không tìm thấy gốc Stocks/notebooks có outputs Bước 03 và Bước 07. '
        f'CWD hiện tại: {Path.cwd()}.'
    )

# Nếu có nhiều bản copy cùng tên, buộc chọn tệp có SHA-256 khớp manifest.
def find_file(label, candidates, expected_sha=None):
    candidates = [ROOT / x for x in candidates]
    existent = [p for p in candidates if p.is_file()]
    if not existent:
        raise FileNotFoundError(f'Thiếu {label}. Đã tìm tại: ' + ', '.join(map(str, candidates)))
    if expected_sha:
        matched = [p for p in existent if sha256_file(p) == expected_sha]
        if not matched:
            raise ValueError(f'{label}: không tệp nào khớp SHA-256 đã khóa. Có thể nhầm phiên bản.')
        return matched[0]
    return existent[0]

POLICY_PATH = find_file('frozen policy P0', [
    '07_A_P0_frozen_policy(1).json',
    'outputs/07_P0_final_pretest_A/07_A_P0_frozen_policy.json',
    '07_A_P0_frozen_policy.json'
])
policy = json.loads(POLICY_PATH.read_text(encoding='utf-8-sig'))
assert policy['experiment'] == 'EXP1_A_P0'
assert policy['status'] == 'READY_FOR_SEPARATE_TEST_EVALUATION'
assert policy['model_features'] == FEATURES
assert policy['feature_window'] == 60
assert policy['calibration_quantile'] == 0.99
assert policy['test_scored'] is False
assert policy['models_refit'] is False
assert policy['proxy_name'] == 'P0_final_std20_mean20_OR'

GATE_PATH = find_file('pre-test gate', [
    '07_pretest_gate(1).json',
    'outputs/07_P0_final_pretest_A/07_pretest_gate.json',
    '07_pretest_gate.json'
])
gate = json.loads(GATE_PATH.read_text(encoding='utf-8-sig'))
required_gate_flags = [
    'data_03_ready_and_chronological', 'proxy_valid_all_model_ready_rows',
    'proxy_past_only', 'checkpoint_files_match_step07_hash',
    'checkpoint_scores_reproduce_step065', 'step07_q99_reproduced_without_refit',
    'q99_unchanged', 'test_open_authorized'
]
for flag in required_gate_flags:
    assert gate.get(flag) is True, f'Pre-test gate chưa đạt: {flag}'
assert gate['test_scores_computed'] is False

THRESH_PATH = find_file('07_thresholds.json', [
    'outputs/07_threshold_calibration_A/07_thresholds.json',
    '07_thresholds(1).json', '07_thresholds.json'
], policy['step07_threshold_manifest_sha256'])
threshold_manifest = json.loads(THRESH_PATH.read_text(encoding='utf-8-sig'))
assert threshold_manifest['quantile'] == 0.99
assert threshold_manifest['rule'] == 'score > threshold'
assert threshold_manifest['n_calibration'] == 962
assert threshold_manifest['feature_cols'] == FEATURES
THRESHOLDS = {m: float(threshold_manifest['thresholds'][m]['threshold']) for m in MODEL_ORDER}
for m in MODEL_ORDER:
    assert np.isclose(THRESHOLDS[m], policy['thresholds'][m], rtol=0, atol=1e-12)

In [4]:
PATHS = {
    'full': find_file('03_features_all.csv', [
        'outputs/03_feature_engineering/03_features_all.csv', '03_features_all.csv'
    ], policy['feature_data_sha256']),
    'dev': find_file('03_model_ready_development.csv', [
        'outputs/03_feature_engineering/03_model_ready_development.csv',
        '03_model_ready_development.csv'
    ], policy['development_data_sha256']),
    'feature_config': find_file('03_feature_config.json', [
        'outputs/03_feature_engineering/03_feature_config.json', '03_feature_config.json'
    ], policy['feature_config_sha256']),
    'val_select_baseline': find_file('06_5_aligned_val_scores.csv', [
        'outputs/06_5_model_comparison_A/06_5_aligned_val_scores.csv',
        '06_5_aligned_val_scores.csv'
    ], policy['val_select_score_sha256']),
    'IF': find_file('05_if.joblib', [
        'outputs/05_model_training_A/05_if.joblib', '05_if.joblib'
    ], policy['model_sha256']['IF']),
    'AE': find_file('AE checkpoint epoch1495', [
        'outputs/06_2_ae_validation_A/06_2_ae_seed42_lr001_epoch1495_best.pt',
        '06_2_ae_seed42_lr001_epoch1495_best.pt'
    ], policy['model_sha256']['AE']),
    'LOF': find_file('LOF k20 novelty', [
        'outputs/06_3_lof_validation_A/06_3_lof_baseline_k20_euclidean.joblib',
        '06_3_lof_baseline_k20_euclidean.joblib'
    ], policy['model_sha256']['LOF'])
}
config = json.loads(PATHS['feature_config'].read_text(encoding='utf-8'))
assert config['features_for_model'] == FEATURES
assert config['window_prior_valid_feature_values_per_ticker'] == 60
assert float(config['mad_normal_factor']) == 1.4826
assert float(config['train_floor_quantile_positive_mad_scales']) == 0.1
floors = config['train_floors']
assert list(floors) == ['gap','intraday','range','volume']
assert all(float(v) > 0 for v in floors.values())

dev = pd.read_csv(PATHS['dev'], encoding='utf-8-sig')
assert len(dev) == 12779
assert not (dev['split'] == 'test').any()
assert int((dev['split'] == 'train').sum()) == 10387
assert int(((dev['split'] == 'val') & (dev['val_role'] == 'select')).sum()) == 1430
assert int(((dev['split'] == 'val') & (dev['val_role'] == 'calibrate')).sum()) == 962
assert np.isfinite(dev[FEATURES].to_numpy(float)).all()

In [5]:
torch.set_num_threads(1)

class SmallAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 2), nn.Tanh(), nn.Linear(2, 4))

    def forward(self, x):
        return self.net(x)

@torch.no_grad()
def score_ae(model, x):
    t = torch.from_numpy(np.array(x, dtype=np.float32, copy=True))
    model.eval()
    return (model(t) - t).square().mean(dim=1).cpu().numpy().astype(float)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always', InconsistentVersionWarning)
    if_model = joblib.load(PATHS['IF'])
    lof_model = joblib.load(PATHS['LOF'])
    version_messages = sorted(set(str(w.message) for w in caught
                                  if issubclass(w.category, InconsistentVersionWarning)))
assert isinstance(if_model, IsolationForest)
assert if_model.n_features_in_ == 4 and if_model.n_estimators == 200
assert if_model.max_samples == 256 and if_model.random_state == 42
assert if_model.max_features == 1.0 and if_model.contamination == 'auto'
assert isinstance(lof_model, LocalOutlierFactor)
assert lof_model.n_features_in_ == 4 and lof_model.n_neighbors == 20
assert lof_model.novelty is True and lof_model.metric == 'minkowski'
assert lof_model.p == 2

ckpt = torch.load(PATHS['AE'], map_location='cpu', weights_only=True)
assert ckpt['features'] == FEATURES and ckpt['architecture'] == '4-2-4'
assert ckpt['seed'] == 42 and ckpt['best_epoch'] == 1495
assert abs(ckpt['learning_rate'] - 0.001) < 1e-10
ae_model = SmallAE()
ae_model.load_state_dict(ckpt['state_dict'], strict=True)
ae_model.eval()

def score_models(x):
    """Output score direction ALWAYS higher_is_more_anomalous."""
    return {
        'IF': -if_model.score_samples(x),
        'AE': score_ae(ae_model, x),
        'LOF': -lof_model.score_samples(x),
    }

baseline = pd.read_csv(PATHS['val_select_baseline'], encoding='utf-8-sig')
baseline['date'] = pd.to_datetime(baseline['date'], format='%Y-%m-%d')
baseline = baseline.sort_values(['date', 'ticker']).reset_index(drop=True)
assert len(baseline) == 1430
assert not baseline.duplicated(['ticker','date']).any()
val_select = dev.loc[(dev['split'] == 'val') & (dev['val_role'] == 'select')].copy()
val_select['date'] = pd.to_datetime(val_select['date'], format='%Y-%m-%d')
val_select = val_select.sort_values(['date', 'ticker']).reset_index(drop=True)
assert val_select[['ticker','date']].equals(baseline[['ticker','date']])
X_select = val_select[FEATURES].to_numpy(dtype=np.float32, copy=True)
score_check = score_models(X_select)
reproduction = []
for name in MODEL_ORDER:
    ref = baseline[f'score_{name.lower()}'].to_numpy(float)
    diff = float(np.max(np.abs(score_check[name] - ref)))
    passed = diff <= TOLERANCES[name]
    reproduction.append({'model': name, 'n': 1430,
                         'max_abs_score_difference': diff,
                         'tolerance': TOLERANCES[name], 'passed': passed})
    assert passed, f'{name}: checkpoint lệch điểm Bước 06.5: {diff}'
print('PASS — Checkpoint reproduction Val-select:')
print(pd.DataFrame(reproduction).to_string(index=False))
if version_messages:
    print('Cảnh báo phiên bản scikit-learn khác khi lưu:', version_messages[0])
    print('Nhưng cả ba mô hình đã qua kiểm tra tái lập điểm Val-select.')

PASS — Checkpoint reproduction Val-select:
model    n  max_abs_score_difference  tolerance  passed
   IF 1430                       0.0    0.00001    True
   AE 1430                       0.0    0.00050    True
  LOF 1430                       0.0    0.00001    True


In [8]:
full = pd.read_csv(PATHS['full'], encoding='utf-8-sig')
full['date'] = pd.to_datetime(full['date'], format='%Y-%m-%d', errors='raise')
assert len(full) == 15977
assert not full.duplicated(['ticker','date']).any()
assert full['ticker'].nunique() == 13
assert len(full.loc[full.split.eq('train')]) == 11180
assert len(full.loc[full.split.eq('val') & full.val_role.eq('select')]) == 1430
assert len(full.loc[full.split.eq('val') & full.val_role.eq('calibrate')]) == 962

test = full.loc[full['split'].eq('test')].copy()
test = test.sort_values(['date','ticker']).reset_index(drop=True)
assert len(test) == N_TEST
assert test['ticker'].nunique() == N_TICKERS
assert test['date'].nunique() == N_TEST_DATES
assert test['date'].min().strftime('%Y-%m-%d') == TEST_FIRST_DATE
assert test['date'].max().strftime('%Y-%m-%d') == TEST_LAST_DATE
assert test.groupby('ticker').size().eq(N_TEST_DATES).all()
assert test['val_role'].eq('not_applicable').all()
assert test['is_model_ready'].eq(True).all() and test['feature_status'].eq('ok').all()
assert np.isfinite(test[FEATURES].to_numpy(float)).all()
assert not test.duplicated(['ticker','date']).any()
assert dev['date'].max() < test['date'].min().strftime('%Y-%m-%d')
assert set(zip(dev.ticker,dev.date)).isdisjoint(set(zip(test.ticker,test.date.dt.strftime('%Y-%m-%d'))))

# Kiểm tra độc lập rằng mọi z_* trên Test tính đúng từ OHLCV và 60 phiên
# TRƯỚC, với floor đã học từ Train. Không fit lại thông số gì trên Test.
component_order = ['gap','intraday','range','volume']
floor_array = np.array([float(floors[k]) for k in component_order])
max_feature_error = 0.0
n_independently_checked = 0
for ticker, g in full.groupby('ticker', sort=False):
    g = g.sort_values('date').reset_index(drop=True)
    prev_close = g['close'].shift(1).to_numpy(float)
    raw = np.column_stack([
        np.log(g['open'].to_numpy(float) / prev_close),
        np.log(g['close'].to_numpy(float) / g['open'].to_numpy(float)),
        np.log(g['high'].to_numpy(float) / g['low'].to_numpy(float)),
        np.log1p(g['volume'].to_numpy(float)),
    ])
    for i in np.flatnonzero(g.split.eq('test').to_numpy()):
        hist = raw[i-60:i]
        assert hist.shape == (60,4) and np.isfinite(hist).all()
        median = np.median(hist, axis=0)
        mad = np.median(np.abs(hist - median), axis=0)
        expected = (raw[i] - median) / np.maximum(1.4826 * mad, floor_array)
        actual = g.loc[i, FEATURES].to_numpy(dtype=float)
        error = float(np.max(np.abs(expected - actual)))
        max_feature_error = max(max_feature_error,error)
        assert np.allclose(expected, actual, rtol=1e-9, atol=1e-8), (
            f'Sai đặc trưng quá khứ tại {ticker}, {g.loc[i,"date"]}, diff={error}'
        )
        n_independently_checked += 1
assert n_independently_checked == N_TEST
print(f'PASS — Dữ liệu Test: {N_TEST} phiên, {N_TICKERS} mã, {N_TEST_DATES} ngày')
print('PASS — Kiểm tra nhân quả W60 + floor Train cho toàn Test, max |delta z| =',
      max_feature_error)

PASS — Dữ liệu Test: 2405 phiên, 13 mã, 185 ngày
PASS — Kiểm tra nhân quả W60 + floor Train cho toàn Test, max |delta z| = 0.0


In [9]:
X_test = np.array(test[FEATURES], dtype=np.float32, copy=True)
score_test = score_models(X_test)
OUT_COLS = ['ticker','date','open','high','low','close','volume',*FEATURES]
predictions = test[OUT_COLS].copy()
for name in MODEL_ORDER:
    key = name.lower()
    scores = np.asarray(score_test[name], dtype=float)
    assert scores.shape == (N_TEST,) and np.isfinite(scores).all()
    predictions[f'score_{key}'] = scores
    predictions[f'{key}_prediction'] = (scores > THRESHOLDS[name]).astype('int8')
predictions['if_main_prediction'] = predictions['if_prediction']
pred_cols = [f'{name.lower()}_prediction' for name in MODEL_ORDER]
predictions['n_models_alert'] = predictions[pred_cols].sum(axis=1).astype('int8')
predictions['any_model_alert'] = (predictions['n_models_alert'] > 0).astype('int8')
predictions['all_three_alert'] = (predictions['n_models_alert'] == 3).astype('int8')

# Dùng nhãn 0/1 đã khóa, KHÔNG dùng Proxy P0 để điều chỉnh bất kỳ dự đoán nào.
def alert_pattern(row):
    found = [name for name in MODEL_ORDER if row[f'{name.lower()}_prediction'] == 1]
    return '_'.join(found) if found else 'none'
predictions['alert_pattern'] = predictions.apply(alert_pattern, axis=1)
predictions['date'] = predictions['date'].dt.strftime('%Y-%m-%d')
assert len(predictions) == N_TEST
assert predictions[pred_cols].isin([0,1]).all().all()
for name in MODEL_ORDER:
    col = name.lower()
    np.testing.assert_array_equal(predictions[f'{col}_prediction'].to_numpy(),
                                  (predictions[f'score_{col}'].to_numpy() > THRESHOLDS[name]).astype('int8'))

In [10]:
OUTPUT_DIR = ROOT / 'outputs' / '08_test_evaluation_A'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEST_FILE = OUTPUT_DIR / '08_test_predictions_A.csv'
ALERTS_FILE = OUTPUT_DIR / '08_test_IF_alerts_A.csv'
THRESH_FILE = OUTPUT_DIR / '08_locked_thresholds_used.csv'
QA_FILE = OUTPUT_DIR / '08_test_QA_manifest.json'
REPORT_FILE = OUTPUT_DIR / '08_test_QA_report.md'

predictions.to_csv(TEST_FILE, index=False, encoding='utf-8-sig', float_format='%.12g')
predictions.loc[predictions.if_prediction.eq(1)].to_csv(
    ALERTS_FILE, index=False, encoding='utf-8-sig', float_format='%.12g'
)
pd.DataFrame([
    {'model':m, 'role':'main' if m=='IF' else 'comparison',
     'q':0.99, 'threshold':THRESHOLDS[m], 'comparison_rule':'score > threshold'}
    for m in MODEL_ORDER
]).to_csv(THRESH_FILE,index=False,encoding='utf-8-sig',float_format='%.16g')

saved = pd.read_csv(TEST_FILE, encoding='utf-8-sig')
assert len(saved) == N_TEST
assert saved[['ticker','date']].equals(predictions[['ticker','date']])
assert not saved.duplicated(['ticker','date']).any()
for name in MODEL_ORDER:
    key = name.lower()
    assert np.allclose(saved[f'score_{key}'],predictions[f'score_{key}'],atol=1e-8,rtol=1e-10)
    np.testing.assert_array_equal(saved[f'{key}_prediction'],predictions[f'{key}_prediction'])
assert len(pd.read_csv(ALERTS_FILE)) == int(saved['if_prediction'].sum())

qa = {
    'step':'08_test_evaluation_A',
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'test_first_date':TEST_FIRST_DATE,
    'test_last_date':TEST_LAST_DATE,
    'n_test':N_TEST,
    'n_tickers':N_TICKERS,
    'n_test_dates':N_TEST_DATES,
    'test_has_duplicate_ticker_date':False,
    'test_four_features_finite':True,
    'feature_recomputed_using_only_60_past_sessions':True,
    'feature_max_abs_replay_diff':max_feature_error,
    'feature_config_sha256':sha256_file(PATHS['feature_config']),
    'features_all_sha256':sha256_file(PATHS['full']),
    'model_sha256':{m:sha256_file(PATHS[m]) for m in MODEL_ORDER},
    'threshold_manifest_sha256':sha256_file(THRESH_PATH),
    'frozen_policy_sha256':sha256_file(POLICY_PATH),
    'pretest_gate_file_sha256':sha256_file(GATE_PATH),
    'thresholds':THRESHOLDS,
    'quantile_policy':'Q99 on Val-calibrate ONLY, locked before Test',
    'rule':'score > threshold',
    'score_higher_is_more_anomalous':True,
    'reproduction_val_select':reproduction,
    'reproduction_calibration_thresholds':threshold_check,
    'sklearn_runtime_version':sklearn.__version__,
    'torch_runtime_version':torch.__version__,
    'python_runtime_version':platform.python_version(),
    'sklearn_version_warning_count':len(version_messages),
    'sklearn_version_warning_sample':version_messages[:1],
    'no_model_refit':True,
    'no_feature_scaler_refit':True,
    'no_threshold_recalibration_on_test':True,
    'proxy_p0_used_in_scoring':False,
    'proxy_p0_evaluation_deferred_to_step09':True,
    'no_news_used':True,
    'test_scored':True,
    'result_files':{
        'all_predictions':TEST_FILE.name,
        'if_main_alerts':ALERTS_FILE.name,
        'locked_thresholds_used':THRESH_FILE.name,
    },
    'output_predictions_sha256':sha256_file(TEST_FILE),
}
QA_FILE.write_text(json.dumps(qa,ensure_ascii=False,indent=2),encoding='utf-8')

# Chỉ báo cáo số lượng, KHÔNG dùng số lượng này để chỉnh mô hình/ngưỡng.
counts = {m:int(predictions[f'{m.lower()}_prediction'].sum()) for m in MODEL_ORDER}
lines = [
    '# Bước 08 — Test Evaluation, phương án A',
    '',
    '## Phạm vi',
    f'- Test: {N_TEST} phiên, {N_TICKERS} mã, {N_TEST_DATES} ngày ({TEST_FIRST_DATE} đến {TEST_LAST_DATE}).',
    '- Đầu vào mô hình: z_gap, z_intraday, z_range, z_volume; W60 và floor Train gốc.',
    '- IF chính; AE và LOF đối chiếu. Checkpoint đã đối chiếu trên Val-select.',
    '- Không huấn luyện lại, không dùng Proxy P0 hoặc Test để tính Q99.',
    '',
    '## Ngưỡng Q99 đã khóa trên Val-calibrate',
    *[f'- {m}: {THRESHOLDS[m]:.12g}' for m in MODEL_ORDER],
    '',
    '## Số phiên được mô hình cảnh báo trên Test (không phải số phiên bất thường thật)',
    *[f'- {m}: {counts[m]}/{N_TEST} ({counts[m]/N_TEST*100:.2f}%).' for m in MODEL_ORDER],
    f'- Có ít nhất một cảnh báo: {int(predictions.any_model_alert.sum())}/{N_TEST}.',
    f'- Cả ba mô hình cảnh báo: {int(predictions.all_three_alert.sum())}/{N_TEST}.',
    '',
    '## Kết quả kiểm tra',
    '- Đã đối chiếu SHA-256 đầu vào, checkpoint và ngưỡng trước khi mở Test.',
    '- Đã tính lại bốn đặc trưng Test độc lập từ 60 phiên trước, không dùng thông tin tương lai.',
    '- Không có giá trị thiếu ở đặc trưng, điểm hoặc nhãn; không trùng ticker/date.',
    '- Các ngưỡng không bị thay đổi để ép tỷ lệ cảnh báo Test về 1%.',
    '- Chưa đối chiếu Proxy P0/ROC-AUC/AP: đó là Bước 09.',
    '',
    '## Đầu ra',
    f'- {TEST_FILE.name}: điểm và dự đoán của từng mã-ngày Test.',
    f'- {ALERTS_FILE.name}: các phiên được IF cảnh báo.',
    f'- {THRESH_FILE.name}: ngưỡng cố định đã sử dụng.',
    f'- {QA_FILE.name}: dấu vết cấu hình/checksum và kiểm tra kỹ thuật.',
]
REPORT_FILE.write_text('\n'.join(lines)+'\n',encoding='utf-8')
print('\nPASS: BƯỚC 08 ĐÃ CHẤM TEST BẰNG CẤU HÌNH A CỐ ĐỊNH')
print('Các tệp đã lưu:', OUTPUT_DIR.resolve())
print('Số phiên Test:',N_TEST)
print('Số cảnh báo [IF, AE, LOF]:',counts)
print('Không sử dụng Proxy P0 hoặc thay đổi ngưỡng. Đối chiếu P0 ở Bước 09.')


PASS: BƯỚC 08 ĐÃ CHẤM TEST BẰNG CẤU HÌNH A CỐ ĐỊNH
Các tệp đã lưu: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\08_test_evaluation_A
Số phiên Test: 2405
Số cảnh báo [IF, AE, LOF]: {'IF': 13, 'AE': 21, 'LOF': 31}
Không sử dụng Proxy P0 hoặc thay đổi ngưỡng. Đối chiếu P0 ở Bước 09.
